# 12 — Ex-Ante Rolling Beta Estimation

Estimate monthly stock betas under the pre-specified CAPM methodology. This notebook validates beta estimates only; portfolio beta-neutralisation is deferred until this gate passes.

**Locked specification:** 252 CRSP market trading days, skip 5 market trading days immediately before formation, minimum 126 valid paired observations, daily excess-return OLS, no winsorisation or imputation.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(ROOT))

from src.clean_monthly import CleaningConfig, build_clean_monthly_panel
from src.daily_data import read_daily_stock, read_daily_market, read_fama_french_daily_rf, align_market_and_rf
from src.beta import BetaConfig, estimate_monthly_capm_beta, beta_distribution


## 1. Paths and inputs


In [ ]:
MONTHLY = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'
DAILY_STOCK = ROOT / 'data/raw/daily_stock_naq_1998_2025_prod_v1.csv.gz'
DAILY_MARKET = ROOT / 'data/raw/daily_crsp_vw_market_1998_2025_v1.csv.gz'
FF_DAILY = ROOT / 'data/raw/F-F_Research_Data_Factors_daily.csv'

for p in [MONTHLY, DAILY_STOCK, DAILY_MARKET, FF_DAILY]:
    print(p.name, '->', p.exists())


## 2. Build formation universe and aligned daily inputs


In [ ]:
clean, _ = build_clean_monthly_panel(MONTHLY, CleaningConfig())
daily_stock = read_daily_stock(DAILY_STOCK)
market = read_daily_market(DAILY_MARKET)
rf = read_fama_french_daily_rf(FF_DAILY)
market_rf, alignment_audit = align_market_and_rf(market, rf)

print('Formation rows:', len(clean))
print('Daily stock rows after exact dedup:', len(daily_stock))
print('Aligned market dates:', len(market_rf))


## 3. Estimate monthly ex-ante CAPM beta


In [ ]:
config = BetaConfig(lookback_days=252, skip_days=5, min_obs=126)

beta, beta_audit = estimate_monthly_capm_beta(
    clean[['PERMNO','MthCalDt']],
    daily_stock,
    market_rf,
    config=config,
)

beta.head()


## 4. Overall beta distribution


In [ ]:
beta_distribution(beta)


## 5. Coverage over time


In [ ]:
beta_audit[['ValidBetaPct','MeanObs','MedianObs']].describe(percentiles=[0.01,0.05,0.5,0.95,0.99])


In [ ]:
check_months = ['2000-01','2008-12','2020-12','2025-12']
beta_audit[
    beta_audit['MthCalDt'].dt.to_period('M').astype(str).isin(check_months)
]


## 6. Extreme beta diagnostics — inspect, do not winsorise


In [ ]:
valid = beta.dropna(subset=['Beta']).copy()
print('Lowest betas')
display(valid.nsmallest(15, 'Beta')[['PERMNO','MthCalDt','Beta','BetaNObs']])
print('Highest betas')
display(valid.nlargest(15, 'Beta')[['PERMNO','MthCalDt','Beta','BetaNObs']])


## 7. Basic consistency checks


In [ ]:
assert not beta.duplicated(['PERMNO','MthCalDt']).any()
assert (beta.loc[beta['Beta'].notna(), 'BetaNObs'] >= 126).all()
assert beta['BetaNObs'].max() <= 252

print('Unique PERMNO-month beta keys: PASS')
print('Minimum-observation rule: PASS')
print('Fixed 252-market-day window bound: PASS')


## Methodology gate before portfolio neutralisation

Proceed only after checking:
- beta coverage is high and stable enough for the intended comparator;
- the beta distribution is economically plausible;
- extreme beta estimates can be explained by data/history rather than coding errors;
- all valid estimates respect the 126-observation minimum and fixed market-calendar window.

Do **not** change the 252/5/126 specification after seeing these results.
